# Notebook 1 — Read and Join the Olist Tables

**Author:** Haneen Mohammad Dahbour

## Purpose

Build one analysis-ready machine-learning table from the validated Olist PostgreSQL tables.

## Prediction problem

Predict whether an order will be delivered after its estimated delivery date.

## Input

The nine validated Olist tables stored in the local PostgreSQL database named `olist`.

## Output artifact

`../data/processed/orders_joined.parquet`

The artifact must contain exactly one row per `order_id`.

## Unit of analysis

**One row represents one customer order.**

## Rules for this notebook

- Inspect every source table before joining.
- Confirm what one row represents in every table.
- Aggregate one-to-many tables before joining them.
- Keep only columns required for later analysis and modelling.
- Do not create the late-delivery target here; that belongs in Notebook 2.
- Validate row count and `order_id` uniqueness after building the final table.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine, inspect, text

print(f"NumPy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"SQLAlchemy version: {sqlalchemy.__version__}")
print("Imports completed successfully.")

NumPy version: 2.5.2
pandas version: 3.0.5
SQLAlchemy version: 2.0.52
Imports completed successfully.


In [2]:
NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == "notebooks"
    else NOTEBOOK_DIR
)

assert (PROJECT_ROOT / "docker-compose.yml").exists(), (
    "Project root could not be identified."
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = PROCESSED_DATA_DIR / "orders_joined.parquet"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Processed-data directory: {PROCESSED_DATA_DIR}")
print(f"Output artifact: {OUTPUT_PATH}")

Notebook directory: D:\Documents\mlops-olist\notebooks
Project root: D:\Documents\mlops-olist
Processed-data directory: D:\Documents\mlops-olist\data\processed
Output artifact: D:\Documents\mlops-olist\data\processed\orders_joined.parquet


## 1. Connect to PostgreSQL

Create a SQLAlchemy engine and verify the database connection using a small read-only query.

For reproducible automated execution, the notebook first reads the password from the `OLIST_DB_PASSWORD` environment variable. If the variable is unavailable, it securely prompts the user without displaying or storing the password.


In [3]:
from getpass import getpass
from sqlalchemy import URL

db_password = os.getenv("OLIST_DB_PASSWORD")

if db_password:
    password_source = "OLIST_DB_PASSWORD environment variable"
else:
    db_password = getpass("PostgreSQL password: ")
    password_source = "secure interactive prompt"

database_url = URL.create(
    drivername="postgresql+psycopg",
    username="olist_user",
    password=db_password,
    host="localhost",
    port=5432,
    database="olist",
)

engine = create_engine(
    database_url,
    pool_pre_ping=True,
)

with engine.connect() as connection:
    connection_info = connection.execute(
        text(
            """
            SELECT
                current_database() AS database_name,
                current_user AS database_user;
            """
        )
    ).mappings().one()

print(f"Password source: {password_source}")
print(f"Connected database: {connection_info['database_name']}")
print(f"Connected user: {connection_info['database_user']}")
print("PostgreSQL connection succeeded.")


Password source: OLIST_DB_PASSWORD environment variable
Connected database: olist
Connected user: olist_user
PostgreSQL connection succeeded.


## 2. Inspect the source tables

Inspect the size, columns, and primary key of every source table before reading or joining the data.

In [4]:
db_inspector = inspect(engine)

table_names = sorted(
    db_inspector.get_table_names(schema="public")
)

expected_tables = {
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "product_category_translation",
    "products",
    "sellers",
}

print(f"Number of tables: {len(table_names)}")

for table_name in table_names:
    print(f"- {table_name}")

assert set(table_names) == expected_tables, (
    "The PostgreSQL tables do not match the nine expected Olist tables."
)

Number of tables: 9
- customers
- geolocation
- order_items
- order_payments
- order_reviews
- orders
- product_category_translation
- products
- sellers


In [5]:
table_overview = []

with engine.connect() as connection:
    for table_name in table_names:
        row_count = connection.execute(
            text(f'SELECT COUNT(*) FROM "{table_name}"')
        ).scalar_one()

        columns = db_inspector.get_columns(
            table_name,
            schema="public",
        )

        primary_key = db_inspector.get_pk_constraint(
            table_name,
            schema="public",
        ).get("constrained_columns", [])

        table_overview.append(
            {
                "table_name": table_name,
                "row_count": row_count,
                "column_count": len(columns),
                "primary_key": ", ".join(primary_key) or "No primary key",
            }
        )

table_overview_df = pd.DataFrame(table_overview)

table_overview_df

,table_name,row_count,column_count,primary_key
0,customers,99441,5,customer_id
1,geolocation,1000163,5,No primary key
2,order_items,112650,7,"order_id, order_item_id"
3,order_payments,103886,5,"order_id, payment_sequential"
4,order_reviews,99224,7,"review_id, order_id"
5,orders,99441,8,order_id
6,product_category_translation,71,2,product_category_name
7,products,32951,9,product_id
8,sellers,3095,4,seller_id


### Initial structure findings

- `orders` is the base table because it contains exactly one row per `order_id`.
- `customers`, `products`, `sellers`, and `product_category_translation` have one row per primary-key value.
- `order_items` contains multiple items for some orders and must be aggregated before joining to `orders`.
- `order_payments` contains multiple payment sequences for some orders and must be aggregated before joining.
- `order_reviews` is not guaranteed to contain one row per order and includes information created after purchase or delivery.
- `geolocation` has no primary key and contains repeated ZIP-prefix observations, so it cannot be joined directly.
- The final joined artifact must preserve a maximum of one row per `order_id`.

In [6]:
column_metadata = []

for table_name in table_names:
    primary_key_columns = set(
        db_inspector.get_pk_constraint(
            table_name,
            schema="public",
        ).get("constrained_columns", [])
    )

    columns = db_inspector.get_columns(
        table_name,
        schema="public",
    )

    for position, column in enumerate(columns, start=1):
        column_metadata.append(
            {
                "table_name": table_name,
                "position": position,
                "column_name": column["name"],
                "data_type": str(column["type"]),
                "nullable": column["nullable"],
                "is_primary_key": column["name"] in primary_key_columns,
            }
        )

column_metadata_df = pd.DataFrame(column_metadata)

for table_name in table_names:
    print(f"\nSchema: {table_name}")

    display(
        column_metadata_df.loc[
            column_metadata_df["table_name"] == table_name,
            [
                "position",
                "column_name",
                "data_type",
                "nullable",
                "is_primary_key",
            ],
        ].reset_index(drop=True)
    )


Schema: customers


,position,column_name,data_type,nullable,is_primary_key
0,1,customer_id,CHAR(32),False,True
1,2,customer_unique_id,CHAR(32),False,False
2,3,customer_zip_code_prefix,CHAR(5),False,False
3,4,customer_city,TEXT,False,False
4,5,customer_state,CHAR(2),False,False



Schema: geolocation


,position,column_name,data_type,nullable,is_primary_key
0,1,geolocation_zip_code_prefix,CHAR(5),False,False
1,2,geolocation_lat,DOUBLE PRECISION,False,False
2,3,geolocation_lng,DOUBLE PRECISION,False,False
3,4,geolocation_city,TEXT,False,False
4,5,geolocation_state,CHAR(2),False,False



Schema: order_items


...

,position,column_name,data_type,nullable,is_primary_key
0,1,product_category_name,TEXT,False,True
1,2,product_category_name_english,TEXT,False,False



Schema: products


,position,column_name,data_type,nullable,is_primary_key
0,1,product_id,CHAR(32),False,True
1,2,product_category_name,TEXT,True,False
2,3,product_name_lenght,INTEGER,True,False
3,4,product_description_lenght,INTEGER,True,False
4,5,product_photos_qty,INTEGER,True,False
5,6,product_weight_g,"NUMERIC(12, 2)",True,False
6,7,product_length_cm,"NUMERIC(12, 2)",True,False
7,8,product_height_cm,"NUMERIC(12, 2)",True,False
8,9,product_width_cm,"NUMERIC(12, 2)",True,False



Schema: sellers


,position,column_name,data_type,nullable,is_primary_key
0,1,seller_id,CHAR(32),False,True
1,2,seller_zip_code_prefix,CHAR(5),False,False
2,3,seller_city,TEXT,False,False
3,4,seller_state,CHAR(2),False,False


### Schema findings

- `orders` contains the estimated and actual customer-delivery timestamps needed to construct the target in Notebook 2.
- Actual delivery timestamps and review information are future information and must not be used as model predictors.
- Customer and seller ZIP prefixes are stored as five-character text values, preserving leading zeros.
- Product-category and physical product attributes contain allowed missing values.
- Review comments are optional and contain post-purchase information.
- `order_items`, `order_payments`, and potentially `order_reviews` require order-level aggregation.
- The original product schema misspells `length` as `lenght`; clearer names will be used in the final artifact.

In [7]:
table_samples = {}

for table_name in table_names:
    sample_query = text(
        f'SELECT * FROM "{table_name}" LIMIT 3'
    )

    sample_df = pd.read_sql(
        sample_query,
        con=engine,
    )

    table_samples[table_name] = sample_df

    print(f"\nSample rows: {table_name}")
    display(sample_df)


Sample rows: customers


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP



Sample rows: geolocation


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP



Sample rows: order_items


...

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto



Sample rows: products


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154.0,18.0,9.0,15.0



Sample rows: sellers


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


## 3. Measure table multiplicity

The final dataset must contain exactly one row per order.

The item, payment, and review tables may contain multiple records for one
`order_id`. Their multiplicity must therefore be measured before aggregation
and joining.

In [8]:
child_tables = [
    "order_items",
    "order_payments",
    "order_reviews",
]

multiplicity_results = []

for table_name in child_tables:
    query = text(
        f"""
        SELECT
            COUNT(*) AS orders_present,
            MIN(records_per_order) AS min_records_per_order,
            ROUND(AVG(records_per_order), 2) AS avg_records_per_order,
            MAX(records_per_order) AS max_records_per_order,
            COUNT(*) FILTER (
                WHERE records_per_order > 1
            ) AS orders_with_multiple_records
        FROM (
            SELECT
                order_id,
                COUNT(*) AS records_per_order
            FROM "{table_name}"
            GROUP BY order_id
        ) AS order_counts
        """
    )

    result = pd.read_sql(query, con=engine)
    result.insert(0, "table_name", table_name)
    multiplicity_results.append(result)

multiplicity_df = pd.concat(
    multiplicity_results,
    ignore_index=True,
)

multiplicity_df

,table_name,orders_present,min_records_per_order,avg_records_per_order,max_records_per_order,orders_with_multiple_records
0,order_items,98666,1,1.14,21,9803
1,order_payments,99440,1,1.04,29,2961
2,order_reviews,98673,1,1.01,3,547


## 4. Aggregate order items

The `order_items` table contains one row per item, while the required unit of
analysis is one row per order.

The table is aggregated by `order_id` before joining so item-level records do
not duplicate the final order rows.

In [9]:
items_order_level = pd.read_sql(
    text(
        """
        SELECT
            order_id,
            COUNT(*) AS item_count,
            COUNT(DISTINCT product_id) AS unique_product_count,
            COUNT(DISTINCT seller_id) AS unique_seller_count,

            SUM(price)::DOUBLE PRECISION AS item_price_total,
            AVG(price)::DOUBLE PRECISION AS item_price_mean,
            MAX(price)::DOUBLE PRECISION AS item_price_max,

            SUM(freight_value)::DOUBLE PRECISION AS freight_value_total,
            AVG(freight_value)::DOUBLE PRECISION AS freight_value_mean,

            MIN(shipping_limit_date) AS shipping_limit_date_min,
            MAX(shipping_limit_date) AS shipping_limit_date_max

        FROM order_items
        GROUP BY order_id
        """
    ),
    con=engine,
)

print("Original item rows:", 112_650)
print("Aggregated order rows:", len(items_order_level))
print(
    "Duplicated order IDs:",
    items_order_level["order_id"].duplicated().sum(),
)

display(items_order_level.head())

Original item rows: 112650
Aggregated order rows: 98666
Duplicated order IDs: 0


,order_id,item_count,unique_product_count,unique_seller_count,item_price_total,item_price_mean,item_price_max,freight_value_total,freight_value_mean,shipping_limit_date_min,shipping_limit_date_max
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,58.90,58.90,13.29,13.29,2017-09-19 09:45:35,2017-09-19 09:45:35
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,239.90,239.90,19.93,19.93,2017-05-03 11:05:13,2017-05-03 11:05:13
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,199.00,199.00,17.87,17.87,2018-01-18 14:48:30,2018-01-18 14:48:30
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.99,12.99,12.79,12.79,2018-08-15 10:10:18,2018-08-15 10:10:18
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,199.90,199.90,18.14,18.14,2017-02-13 13:57:51,2017-02-13 13:57:51


## 5. Enrich order items with product information

Each item references a product through `product_id`.

Product attributes and English category names are attached at item level before
being aggregated to one row per order. Left joins are used so an item is not
removed when optional product or translation information is missing.

In [10]:
item_product_details = pd.read_sql(
    text(
        """
        SELECT
            oi.order_id,
            oi.order_item_id,
            oi.product_id,

            p.product_id IS NOT NULL AS product_record_found,

            COALESCE(
                pct.product_category_name_english,
                p.product_category_name,
                'unknown'
            ) AS product_category,

            p.product_name_lenght,
            p.product_description_lenght,
            p.product_photos_qty,

            p.product_weight_g::DOUBLE PRECISION AS product_weight_g,
            p.product_length_cm::DOUBLE PRECISION AS product_length_cm,
            p.product_height_cm::DOUBLE PRECISION AS product_height_cm,
            p.product_width_cm::DOUBLE PRECISION AS product_width_cm

        FROM order_items AS oi

        LEFT JOIN products AS p
            ON oi.product_id = p.product_id

        LEFT JOIN product_category_translation AS pct
            ON p.product_category_name = pct.product_category_name
        """
    ),
    con=engine,
)

print("Enriched item rows:", len(item_product_details))

print(
    "Items without matching product:",
    (~item_product_details["product_record_found"]).sum(),
)

print(
    "Items with unknown category:",
    item_product_details["product_category"].eq("unknown").sum(),
)

display(item_product_details.head())

Enriched item rows: 112650
Items without matching product: 0
Items with unknown category: 1603


,order_id,order_item_id,product_id,product_record_found,product_category,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,True,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,True,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,True,furniture_decor,59.0,695.0,2.0,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,True,perfumery,42.0,480.0,1.0,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,True,garden_tools,59.0,409.0,1.0,3750.0,35.0,40.0,30.0


## 6. Aggregate product characteristics by order

Product details are currently stored at item level. They are summarized by
`order_id` to preserve the required one-row-per-order unit of analysis.

A primary product category is selected as the category occurring most often
among an order's item rows. Alphabetical order is used as a deterministic
tie-breaker.

In [11]:
category_counts = (
    item_product_details
    .groupby(
        ["order_id", "product_category"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "category_item_count"})
)

primary_categories = (
    category_counts
    .sort_values(
        by=[
            "order_id",
            "category_item_count",
            "product_category",
        ],
        ascending=[True, False, True],
    )
    .drop_duplicates(subset="order_id")
    .rename(
        columns={
            "product_category": "primary_product_category",
            "category_item_count": "primary_category_item_count",
        }
    )
)

product_numeric_summary = (
    item_product_details
    .groupby("order_id", as_index=False)
    .agg(
        unique_product_category_count=(
            "product_category",
            "nunique",
        ),
        product_name_length_mean=(
            "product_name_lenght",
            "mean",
        ),
        product_description_length_mean=(
            "product_description_lenght",
            "mean",
        ),
        product_photos_qty_mean=(
            "product_photos_qty",
            "mean",
        ),
        product_weight_g_total=(
            "product_weight_g",
            lambda values: values.sum(min_count=1),
        ),
        product_weight_g_mean=(
            "product_weight_g",
            "mean",
        ),
        product_length_cm_mean=(
            "product_length_cm",
            "mean",
        ),
        product_height_cm_mean=(
            "product_height_cm",
            "mean",
        ),
        product_width_cm_mean=(
            "product_width_cm",
            "mean",
        ),
    )
)

product_order_level = product_numeric_summary.merge(
    primary_categories[
        [
            "order_id",
            "primary_product_category",
            "primary_category_item_count",
        ]
    ],
    on="order_id",
    how="left",
    validate="one_to_one",
)

print("Product order rows:", len(product_order_level))
print(
    "Duplicated order IDs:",
    product_order_level["order_id"].duplicated().sum(),
)
print(
    "Orders with primary category unknown:",
    product_order_level[
        "primary_product_category"
    ].eq("unknown").sum(),
)

display(product_order_level.head())

Product order rows: 98666
Duplicated order IDs: 0
Orders with primary category unknown: 1393


,order_id,unique_product_category_count,product_name_length_mean,product_description_length_mean,product_photos_qty_mean,product_weight_g_total,product_weight_g_mean,product_length_cm_mean,product_height_cm_mean,product_width_cm_mean,primary_product_category,primary_category_item_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.0,598.0,4.0,650.0,650.0,28.0,9.0,14.0,cool_stuff,1
1,00018f77f2f0320c557190d7a144bdd3,1,56.0,239.0,2.0,30000.0,30000.0,50.0,30.0,40.0,pet_shop,1
2,000229ec398224ef6ca0657da4fc703e,1,59.0,695.0,2.0,3050.0,3050.0,33.0,13.0,33.0,furniture_decor,1
3,00024acbcdf0a6daa1e931b038114c75,1,42.0,480.0,1.0,200.0,200.0,16.0,10.0,15.0,perfumery,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,59.0,409.0,1.0,3750.0,3750.0,35.0,40.0,30.0,garden_tools,1


## 7. Combine item and product summaries

The item-price summary and product-characteristic summary both contain exactly
one row per `order_id`.

They can therefore be combined with a validated one-to-one merge without
creating duplicate order rows.

In [12]:
item_features_order_level = items_order_level.merge(
    product_order_level,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator="_product_merge",
)

print(
    "Rows before merge:",
    len(items_order_level),
)

print(
    "Rows after merge:",
    len(item_features_order_level),
)

print(
    "Orders without product summary:",
    item_features_order_level[
        "_product_merge"
    ].eq("left_only").sum(),
)

print(
    "Duplicated order IDs:",
    item_features_order_level[
        "order_id"
    ].duplicated().sum(),
)

assert len(item_features_order_level) == len(items_order_level)

assert (
    item_features_order_level["order_id"]
    .duplicated()
    .sum()
    == 0
)

assert (
    item_features_order_level["_product_merge"]
    .eq("left_only")
    .sum()
    == 0
)

item_features_order_level = (
    item_features_order_level
    .drop(columns="_product_merge")
)

print(
    "Final item-feature shape:",
    item_features_order_level.shape,
)

display(item_features_order_level.head())

Rows before merge: 98666
Rows after merge: 98666
Orders without product summary: 0
Duplicated order IDs: 0
Final item-feature shape: (98666, 22)


,order_id,item_count,unique_product_count,unique_seller_count,item_price_total,item_price_mean,item_price_max,freight_value_total,freight_value_mean,shipping_limit_date_min,...,product_name_length_mean,product_description_length_mean,product_photos_qty_mean,product_weight_g_total,product_weight_g_mean,product_length_cm_mean,product_height_cm_mean,product_width_cm_mean,primary_product_category,primary_category_item_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,58.90,58.90,13.29,13.29,2017-09-19 09:45:35,...,58.0,598.0,4.0,650.0,650.0,28.0,9.0,14.0,cool_stuff,1
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,239.90,239.90,19.93,19.93,2017-05-03 11:05:13,...,56.0,239.0,2.0,30000.0,30000.0,50.0,30.0,40.0,pet_shop,1
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,199.00,199.00,17.87,17.87,2018-01-18 14:48:30,...,59.0,695.0,2.0,3050.0,3050.0,33.0,13.0,33.0,furniture_decor,1
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.99,12.99,12.79,12.79,2018-08-15 10:10:18,...,42.0,480.0,1.0,200.0,200.0,16.0,10.0,15.0,perfumery,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,199.90,199.90,18.14,18.14,2017-02-13 13:57:51,...,59.0,409.0,1.0,3750.0,3750.0,35.0,40.0,30.0,garden_tools,1


## 8. Aggregate seller information by order

Each order item references a seller through `seller_id`.

Because an order may contain items from multiple sellers, seller information
must be aggregated before joining it to the order-level dataset.

The primary seller is the seller associated with the most item rows in the
order. Seller ID is used as a deterministic tie-breaker.

In [13]:
seller_item_details = pd.read_sql(
    text(
        """
        SELECT
            oi.order_id,
            oi.order_item_id,
            oi.seller_id,

            s.seller_id IS NOT NULL AS seller_record_found,
            s.seller_zip_code_prefix,
            s.seller_city,
            s.seller_state

        FROM order_items AS oi

        LEFT JOIN sellers AS s
            ON oi.seller_id = s.seller_id
        """
    ),
    con=engine,
)

seller_counts = (
    seller_item_details
    .groupby(
        [
            "order_id",
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "seller_item_count"})
)

primary_sellers = (
    seller_counts
    .sort_values(
        by=[
            "order_id",
            "seller_item_count",
            "seller_id",
        ],
        ascending=[True, False, True],
    )
    .drop_duplicates(subset="order_id")
    .rename(
        columns={
            "seller_id": "primary_seller_id",
            "seller_zip_code_prefix": "primary_seller_zip_code_prefix",
            "seller_city": "primary_seller_city",
            "seller_state": "primary_seller_state",
            "seller_item_count": "primary_seller_item_count",
        }
    )
)

seller_diversity = (
    seller_item_details
    .groupby("order_id", as_index=False)
    .agg(
        unique_seller_city_count=(
            "seller_city",
            "nunique",
        ),
        unique_seller_state_count=(
            "seller_state",
            "nunique",
        ),
    )
)

seller_order_level = seller_diversity.merge(
    primary_sellers[
        [
            "order_id",
            "primary_seller_id",
            "primary_seller_zip_code_prefix",
            "primary_seller_city",
            "primary_seller_state",
            "primary_seller_item_count",
        ]
    ],
    on="order_id",
    how="left",
    validate="one_to_one",
)

print("Seller-item rows:", len(seller_item_details))

print(
    "Items without matching seller:",
    (~seller_item_details["seller_record_found"]).sum(),
)

print(
    "Seller order rows:",
    len(seller_order_level),
)

print(
    "Duplicated order IDs:",
    seller_order_level["order_id"].duplicated().sum(),
)

display(seller_order_level.head())

Seller-item rows: 112650
Items without matching seller: 0
Seller order rows: 98666
Duplicated order IDs: 0


,order_id,unique_seller_city_count,unique_seller_state_count,primary_seller_id,primary_seller_zip_code_prefix,primary_seller_city,primary_seller_state,primary_seller_item_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,48436dade18ac8b2bce089ec2a041202,27277,volta redonda,SP,1
1,00018f77f2f0320c557190d7a144bdd3,1,1,dd7ddc04e1b6c2c614352b383efe2d36,03471,sao paulo,SP,1
2,000229ec398224ef6ca0657da4fc703e,1,1,5b51032eddd242adc84c38acab88f23d,37564,borda da mata,MG,1
3,00024acbcdf0a6daa1e931b038114c75,1,1,9d7a1d34a5052409006425275ba1c2b4,14403,franca,SP,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,df560393f3a51e74553ab94004ba5c87,87900,loanda,PR,1


## 9. Attach seller information to item features

The seller summary and combined item-product summary both contain one row per
represented order. They are joined using a validated one-to-one merge.

In [14]:
item_seller_features = item_features_order_level.merge(
    seller_order_level,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator="_seller_merge",
)

print(
    "Rows before seller merge:",
    len(item_features_order_level),
)

print(
    "Rows after seller merge:",
    len(item_seller_features),
)

print(
    "Orders without seller summary:",
    item_seller_features[
        "_seller_merge"
    ].eq("left_only").sum(),
)

print(
    "Duplicated order IDs:",
    item_seller_features[
        "order_id"
    ].duplicated().sum(),
)

assert len(item_seller_features) == len(
    item_features_order_level
)

assert (
    item_seller_features["order_id"]
    .duplicated()
    .sum()
    == 0
)

assert (
    item_seller_features["_seller_merge"]
    .eq("left_only")
    .sum()
    == 0
)

item_seller_features = item_seller_features.drop(
    columns="_seller_merge"
)

print(
    "Item-seller feature shape:",
    item_seller_features.shape,
)

display(item_seller_features.head())

Rows before seller merge: 98666
Rows after seller merge: 98666
Orders without seller summary: 0
Duplicated order IDs: 0
Item-seller feature shape: (98666, 29)


,order_id,item_count,unique_product_count,unique_seller_count,item_price_total,item_price_mean,item_price_max,freight_value_total,freight_value_mean,shipping_limit_date_min,...,product_width_cm_mean,primary_product_category,primary_category_item_count,unique_seller_city_count,unique_seller_state_count,primary_seller_id,primary_seller_zip_code_prefix,primary_seller_city,primary_seller_state,primary_seller_item_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,58.90,58.90,13.29,13.29,2017-09-19 09:45:35,...,14.0,cool_stuff,1,1,1,48436dade18ac8b2bce089ec2a041202,27277,volta redonda,SP,1
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,239.90,239.90,19.93,19.93,2017-05-03 11:05:13,...,40.0,pet_shop,1,1,1,dd7ddc04e1b6c2c614352b383efe2d36,03471,sao paulo,SP,1
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,199.00,199.00,17.87,17.87,2018-01-18 14:48:30,...,33.0,furniture_decor,1,1,1,5b51032eddd242adc84c38acab88f23d,37564,borda da mata,MG,1
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.99,12.99,12.79,12.79,2018-08-15 10:10:18,...,15.0,perfumery,1,1,1,9d7a1d34a5052409006425275ba1c2b4,14403,franca,SP,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,199.90,199.90,18.14,18.14,2017-02-13 13:57:51,...,30.0,garden_tools,1,1,1,df560393f3a51e74553ab94004ba5c87,87900,loanda,PR,1


## 10. Aggregate payment information by order

An order may have multiple payment records, payment sequences, or payment
methods.

Payment records are aggregated to one row per `order_id`. The primary payment
type is defined as the payment method contributing the largest total payment
value to the order.

In [15]:
payment_details = pd.read_sql(
    text(
        """
        SELECT
            order_id,
            payment_sequential,
            payment_type,
            payment_installments,
            payment_value::DOUBLE PRECISION AS payment_value
        FROM order_payments
        """
    ),
    con=engine,
)

payment_type_summary = (
    payment_details
    .groupby(
        ["order_id", "payment_type"],
        as_index=False,
    )
    .agg(
        payment_type_value=(
            "payment_value",
            "sum",
        ),
        payment_type_record_count=(
            "payment_type",
            "size",
        ),
    )
)

primary_payment_types = (
    payment_type_summary
    .sort_values(
        by=[
            "order_id",
            "payment_type_value",
            "payment_type_record_count",
            "payment_type",
        ],
        ascending=[True, False, False, True],
    )
    .drop_duplicates(subset="order_id")
    .rename(
        columns={
            "payment_type": "primary_payment_type",
            "payment_type_value": "primary_payment_type_value",
            "payment_type_record_count": (
                "primary_payment_type_record_count"
            ),
        }
    )
)

payment_numeric_summary = (
    payment_details
    .groupby("order_id", as_index=False)
    .agg(
        payment_record_count=(
            "payment_sequential",
            "size",
        ),
        unique_payment_type_count=(
            "payment_type",
            "nunique",
        ),
        payment_value_total=(
            "payment_value",
            "sum",
        ),
        payment_value_mean=(
            "payment_value",
            "mean",
        ),
        payment_installments_max=(
            "payment_installments",
            "max",
        ),
        payment_installments_mean=(
            "payment_installments",
            "mean",
        ),
    )
)

payment_order_level = payment_numeric_summary.merge(
    primary_payment_types[
        [
            "order_id",
            "primary_payment_type",
            "primary_payment_type_value",
            "primary_payment_type_record_count",
        ]
    ],
    on="order_id",
    how="left",
    validate="one_to_one",
)

print("Original payment rows:", len(payment_details))
print("Payment order rows:", len(payment_order_level))

print(
    "Duplicated order IDs:",
    payment_order_level["order_id"].duplicated().sum(),
)

display(payment_order_level.head())

Original payment rows: 103886
Payment order rows: 99440
Duplicated order IDs: 0


,order_id,payment_record_count,unique_payment_type_count,payment_value_total,payment_value_mean,payment_installments_max,payment_installments_mean,primary_payment_type,primary_payment_type_value,primary_payment_type_record_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,72.19,72.19,2,2.0,credit_card,72.19,1
1,00018f77f2f0320c557190d7a144bdd3,1,1,259.83,259.83,3,3.0,credit_card,259.83,1
2,000229ec398224ef6ca0657da4fc703e,1,1,216.87,216.87,5,5.0,credit_card,216.87,1
3,00024acbcdf0a6daa1e931b038114c75,1,1,25.78,25.78,2,2.0,credit_card,25.78,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,218.04,218.04,3,3.0,credit_card,218.04,1


## 11. Aggregate post-delivery review information

Orders may have multiple reviews, so review records must be aggregated before
joining.

Review information is created after purchase and usually after delivery. These
columns are retained for outcome analysis but are explicitly marked as
post-delivery information and must not be used as prediction-time model
features.

In [16]:
review_order_level = pd.read_sql(
    text(
        """
        SELECT
            order_id,

            COUNT(*) AS post_delivery_review_count,

            AVG(review_score)::DOUBLE PRECISION
                AS post_delivery_review_score_mean,

            MIN(review_score)
                AS post_delivery_review_score_min,

            MAX(review_score)
                AS post_delivery_review_score_max,

            COUNT(*) FILTER (
                WHERE NULLIF(
                    TRIM(review_comment_title),
                    ''
                ) IS NOT NULL
            ) AS post_delivery_review_title_count,

            COUNT(*) FILTER (
                WHERE NULLIF(
                    TRIM(review_comment_message),
                    ''
                ) IS NOT NULL
            ) AS post_delivery_review_message_count,

            AVG(
                LENGTH(review_comment_message)
            ) FILTER (
                WHERE NULLIF(
                    TRIM(review_comment_message),
                    ''
                ) IS NOT NULL
            )::DOUBLE PRECISION
                AS post_delivery_review_message_length_mean,

            MIN(review_creation_date)
                AS post_delivery_review_creation_date_min,

            MAX(review_answer_timestamp)
                AS post_delivery_review_answer_timestamp_max

        FROM order_reviews
        GROUP BY order_id
        """
    ),
    con=engine,
)

print(
    "Original review rows:",
    99_224,
)

print(
    "Review order rows:",
    len(review_order_level),
)

print(
    "Duplicated order IDs:",
    review_order_level["order_id"].duplicated().sum(),
)

display(review_order_level.head())

Original review rows: 99224
Review order rows: 98673
Duplicated order IDs: 0


,order_id,post_delivery_review_count,post_delivery_review_score_mean,post_delivery_review_score_min,post_delivery_review_score_max,post_delivery_review_title_count,post_delivery_review_message_count,post_delivery_review_message_length_mean,post_delivery_review_creation_date_min,post_delivery_review_answer_timestamp_max
0,00010242fe8c5a6d1ba2dd792cb16214,1,5.0,5,5,0,1,46.0,2017-09-21,2017-09-22 10:57:03
1,00018f77f2f0320c557190d7a144bdd3,1,4.0,4,4,0,0,NaN,2017-05-13,2017-05-15 11:34:13
2,000229ec398224ef6ca0657da4fc703e,1,5.0,5,5,0,1,90.0,2018-01-23,2018-01-23 16:06:31
3,00024acbcdf0a6daa1e931b038114c75,1,4.0,4,4,0,0,NaN,2018-08-15,2018-08-15 16:39:01
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,5.0,5,5,0,1,39.0,2017-03-02,2017-03-03 10:54:59


## 12. Aggregate geolocation by ZIP-code prefix

The geolocation table contains multiple coordinate records for the same ZIP
prefix. Joining it directly would duplicate customer and order rows.

A representative latitude and longitude are calculated for each ZIP prefix
using the median, producing exactly one row per ZIP prefix.

In [17]:
geolocation_zip_level = pd.read_sql(
    text(
        """
        SELECT
            geolocation_zip_code_prefix,

            PERCENTILE_CONT(0.5) WITHIN GROUP (
                ORDER BY geolocation_lat
            )::DOUBLE PRECISION AS geolocation_lat_median,

            PERCENTILE_CONT(0.5) WITHIN GROUP (
                ORDER BY geolocation_lng
            )::DOUBLE PRECISION AS geolocation_lng_median,

            COUNT(*) AS geolocation_record_count

        FROM geolocation
        GROUP BY geolocation_zip_code_prefix
        """
    ),
    con=engine,
)

print(
    "Original geolocation rows:",
    1_000_163,
)

print(
    "Unique ZIP-prefix rows:",
    len(geolocation_zip_level),
)

print(
    "Duplicated ZIP prefixes:",
    geolocation_zip_level[
        "geolocation_zip_code_prefix"
    ].duplicated().sum(),
)

assert (
    geolocation_zip_level[
        "geolocation_zip_code_prefix"
    ]
    .duplicated()
    .sum()
    == 0
)

display(geolocation_zip_level.head())

Original geolocation rows: 1000163
Unique ZIP-prefix rows: 19015
Duplicated ZIP prefixes: 0


,geolocation_zip_code_prefix,geolocation_lat_median,geolocation_lng_median,geolocation_record_count
0,01001,-23.550381,-46.634027,26
1,01002,-23.548551,-46.635072,13
2,01003,-23.548977,-46.635313,17
3,01004,-23.549535,-46.634771,22
4,01005,-23.549612,-46.636532,25


## 13. Build the base order and customer table

The `orders` table defines the unit of analysis for the final ML table: one row per order.

Customer attributes are attached with a left join so that every order is preserved, including orders that may not have a matching customer record. At this stage, all order statuses are retained. Decisions about which orders can receive a valid late-delivery label will be made in Notebook 2.

The resulting table must:

- preserve the original number of orders;
- contain no duplicated `order_id` values;
- expose any unmatched customer records;
- retain raw timestamps needed for label creation and later feature engineering.

In [18]:
orders_source_count = pd.read_sql(
    text(
        """
        SELECT COUNT(*) AS order_count
        FROM orders
        """
    ),
    con=engine,
).loc[0, "order_count"]

orders_customer_level = pd.read_sql(
    text(
        """
        SELECT
            o.order_id,
            o.customer_id,
            o.order_status,
            o.order_purchase_timestamp,
            o.order_approved_at,
            o.order_delivered_carrier_date,
            o.order_delivered_customer_date,
            o.order_estimated_delivery_date,

            c.customer_id IS NOT NULL
                AS customer_record_found,

            c.customer_unique_id,
            c.customer_zip_code_prefix,
            c.customer_city,
            c.customer_state

        FROM orders AS o

        LEFT JOIN customers AS c
            ON o.customer_id = c.customer_id
        """
    ),
    con=engine,
)

print("Original order rows:", orders_source_count)
print("Order-customer rows:", len(orders_customer_level))

print(
    "Unique order IDs:",
    orders_customer_level["order_id"].nunique(),
)

print(
    "Duplicated order IDs:",
    orders_customer_level["order_id"].duplicated().sum(),
)

print(
    "Orders without matching customer:",
    (~orders_customer_level["customer_record_found"]).sum(),
)

assert len(orders_customer_level) == orders_source_count

assert (
    orders_customer_level["order_id"]
    .duplicated()
    .sum()
    == 0
)

assert (
    ~orders_customer_level["customer_record_found"]
).sum() == 0

print(
    "Order-customer shape:",
    orders_customer_level.shape,
)

display(orders_customer_level.head())

Original order rows: 99441
Order-customer rows: 99441
Unique order IDs: 99441
Duplicated order IDs: 0
Orders without matching customer: 0
Order-customer shape: (99441, 13)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_record_found,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,7c396fd4830fd04220f754e42b4e5bff,03149,sao paulo,SP
1,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,True,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
2,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,True,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
3,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,True,932afa1e708222e5821dac9cd5db4cae,26525,nilopolis,RJ
4,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,True,39382392765b6dc74812866ee5ee92a7,99655,faxinalzinho,RS


## 14. Attach item, product, and seller features

The previously aggregated item, product, and seller summary is joined to the order-customer base using `order_id`.

A left join is used because the `orders` table defines the final dataset population. Orders without item records must remain visible rather than being silently removed.

The join must:

- preserve all 99,441 orders;
- maintain one row per `order_id`;
- identify orders without an item/seller summary;
- avoid multiplying orders during the join.
  

In [19]:
ml_order_level = orders_customer_level.merge(
    item_seller_features,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator="_item_seller_merge",
)

print(
    "Rows before item/seller merge:",
    len(orders_customer_level),
)

print(
    "Rows after item/seller merge:",
    len(ml_order_level),
)

print(
    "Orders with item/seller summary:",
    ml_order_level[
        "_item_seller_merge"
    ].eq("both").sum(),
)

print(
    "Orders without item/seller summary:",
    ml_order_level[
        "_item_seller_merge"
    ].eq("left_only").sum(),
)

print(
    "Duplicated order IDs:",
    ml_order_level["order_id"].duplicated().sum(),
)

assert len(ml_order_level) == len(orders_customer_level)

assert (
    ml_order_level["order_id"]
    .duplicated()
    .sum()
    == 0
)

orders_without_items_by_status = (
    ml_order_level.loc[
        ml_order_level["_item_seller_merge"].eq("left_only"),
        "order_status",
    ]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

print("\nOrders without item summaries by status:")

display(orders_without_items_by_status)

ml_order_level = ml_order_level.drop(
    columns="_item_seller_merge"
)

print(
    "Current ML-table shape:",
    ml_order_level.shape,
)

display(ml_order_level.head())

Rows before item/seller merge: 99441
Rows after item/seller merge: 99441
Orders with item/seller summary: 98666
Orders without item/seller summary: 775
Duplicated order IDs: 0

Orders without item summaries by status:


,order_status,order_count
0,unavailable,603
1,canceled,164
2,created,5
3,invoiced,2
4,shipped,1


Current ML-table shape: (99441, 41)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_record_found,customer_unique_id,...,product_width_cm_mean,primary_product_category,primary_category_item_count,unique_seller_city_count,unique_seller_state_count,primary_seller_id,primary_seller_zip_code_prefix,primary_seller_city,primary_seller_state,primary_seller_item_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,7c396fd4830fd04220f754e42b4e5bff,...,13.0,housewares,1.0,1.0,1.0,3504c0cb71d7fa48d967e0e4c94d59d9,09350,maua,SP,1.0
1,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,True,3a653a41f6f9fc3d2a113cf8398680e8,...,21.0,auto,1.0,1.0,1.0,4869f7a5dfa277a7dca6462dcf3b52b2,14840,guariba,SP,1.0
2,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,True,7c142cf63193a1473d2e66489a9ae977,...,20.0,pet_shop,1.0,1.0,1.0,66922902710d126a0e7d26b0e3805106,31842,belo horizonte,MG,1.0
3,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,True,932afa1e708222e5821dac9cd5db4cae,...,17.0,auto,1.0,1.0,1.0,16090f2ca825584b5a147ab24aa30c86,12940,atibaia,SP,1.0
4,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,True,39382392765b6dc74812866ee5ee92a7,...,15.0,furniture_decor,1.0,1.0,1.0,63b9ae557efed31d1f7687917d248a8d,13720,sao jose do rio pardo,SP,1.0


### Item and seller join findings

The join preserved all 99,441 orders and maintained one row per order.

A total of 98,666 orders matched an item, product, and seller summary. The remaining 775 orders had no item summary:

- 603 unavailable;
- 164 canceled;
- 5 created;
- 2 invoiced;
- 1 shipped.

Unavailable and canceled orders account for approximately 98.97% of unmatched orders. No delivered orders were found among the unmatched records.

These orders remain in the Notebook 1 artifact. Notebook 2 will define the modelling population based on whether a valid late-delivery label can be calculated.

## 15. Attach payment features

The order-level payment summary is joined to the developing ML table using `order_id`.

The raw payment table cannot be joined directly because one order may contain multiple payment records. Those records were previously aggregated into one row per order.

A left join is used to preserve all orders. The join must:

- retain all 99,441 orders;
- maintain one row per `order_id`;
- identify orders without payment information;
- avoid multiplying orders.

In [20]:
ml_order_level = ml_order_level.merge(
    payment_order_level,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator="_payment_merge",
)

print(
    "Rows before payment merge:",
    len(orders_customer_level),
)

print(
    "Rows after payment merge:",
    len(ml_order_level),
)

print(
    "Orders with payment summary:",
    ml_order_level[
        "_payment_merge"
    ].eq("both").sum(),
)

print(
    "Orders without payment summary:",
    ml_order_level[
        "_payment_merge"
    ].eq("left_only").sum(),
)

print(
    "Duplicated order IDs:",
    ml_order_level["order_id"].duplicated().sum(),
)

assert len(ml_order_level) == len(orders_customer_level)

assert (
    ml_order_level["order_id"]
    .duplicated()
    .sum()
    == 0
)

orders_without_payments_by_status = (
    ml_order_level.loc[
        ml_order_level["_payment_merge"].eq("left_only"),
        "order_status",
    ]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

print("\nOrders without payment summaries by status:")

display(orders_without_payments_by_status)

ml_order_level = ml_order_level.drop(
    columns="_payment_merge"
)

print(
    "Current ML-table shape:",
    ml_order_level.shape,
)

display(ml_order_level.head())

Rows before payment merge: 99441
Rows after payment merge: 99441
Orders with payment summary: 99440
Orders without payment summary: 1
Duplicated order IDs: 0

Orders without payment summaries by status:


,order_status,order_count
0,delivered,1


Current ML-table shape: (99441, 50)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_record_found,customer_unique_id,...,primary_seller_item_count,payment_record_count,unique_payment_type_count,payment_value_total,payment_value_mean,payment_installments_max,payment_installments_mean,primary_payment_type,primary_payment_type_value,primary_payment_type_record_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,7c396fd4830fd04220f754e42b4e5bff,...,1.0,3.0,2.0,38.71,12.903333,1.0,1.0,voucher,20.59,2.0
1,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,True,3a653a41f6f9fc3d2a113cf8398680e8,...,1.0,1.0,1.0,179.12,179.120000,3.0,3.0,credit_card,179.12,1.0
2,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,True,7c142cf63193a1473d2e66489a9ae977,...,1.0,1.0,1.0,72.20,72.200000,1.0,1.0,credit_card,72.20,1.0
3,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,True,932afa1e708222e5821dac9cd5db4cae,...,1.0,1.0,1.0,75.16,75.160000,3.0,3.0,credit_card,75.16,1.0
4,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,True,39382392765b6dc74812866ee5ee92a7,...,1.0,1.0,1.0,35.95,35.950000,1.0,1.0,boleto,35.95,1.0


### Payment join findings

The payment join preserved all 99,441 orders and maintained one row per order.

Payment summaries were found for 99,440 orders. One delivered order did not have a matching payment record. The order is retained because removing it silently would hide a source-data anomaly.

The missing payment values will be handled explicitly during feature engineering. No assumptions are made about the missing payment method or amount.

In [21]:
orders_without_payment_details = (
    ml_order_level.loc[
        ml_order_level["payment_record_count"].isna(),
        [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "item_count",
            "payment_record_count",
            "payment_value_total",
            "primary_payment_type",
        ],
    ]
)

print(
    "Orders with missing payment information:",
    len(orders_without_payment_details),
)

assert len(orders_without_payment_details) == 1

display(orders_without_payment_details)

Orders with missing payment information: 1


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_customer_date,order_estimated_delivery_date,item_count,payment_record_count,payment_value_total,primary_payment_type
15385,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-09 07:47:38,2016-10-04,3.0,NaN,NaN,NaN


## 16. Attach post-delivery review information

The order-level review summary is joined for data-quality analysis and exploratory analysis.

Reviews are post-delivery information. They may help investigate the relationship between delivery performance and customer satisfaction, but they are not eligible prediction-time features.

A left join is used to preserve every order, including orders for which no review was submitted. The join must maintain one row per order and must not multiply the dataset.

In [22]:
ml_order_level = ml_order_level.merge(
    review_order_level,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator="_review_merge",
)

print(
    "Rows before review merge:",
    len(orders_customer_level),
)

print(
    "Rows after review merge:",
    len(ml_order_level),
)

print(
    "Orders with review summary:",
    ml_order_level[
        "_review_merge"
    ].eq("both").sum(),
)

print(
    "Orders without review summary:",
    ml_order_level[
        "_review_merge"
    ].eq("left_only").sum(),
)

print(
    "Duplicated order IDs:",
    ml_order_level["order_id"].duplicated().sum(),
)

assert len(ml_order_level) == len(orders_customer_level)

assert (
    ml_order_level["order_id"]
    .duplicated()
    .sum()
    == 0
)

orders_without_reviews_by_status = (
    ml_order_level.loc[
        ml_order_level["_review_merge"].eq("left_only"),
        "order_status",
    ]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

print("\nOrders without review summaries by status:")

display(orders_without_reviews_by_status)

ml_order_level = ml_order_level.drop(
    columns="_review_merge"
)

print(
    "Current ML-table shape:",
    ml_order_level.shape,
)

display(ml_order_level.head())

Rows before review merge: 99441
Rows after review merge: 99441
Orders with review summary: 98673
Orders without review summary: 768
Duplicated order IDs: 0

Orders without review summaries by status:


,order_status,order_count
0,delivered,646
1,shipped,75
2,canceled,20
3,unavailable,14
4,processing,6
5,invoiced,5
6,created,2


Current ML-table shape: (99441, 59)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_record_found,customer_unique_id,...,primary_payment_type_record_count,post_delivery_review_count,post_delivery_review_score_mean,post_delivery_review_score_min,post_delivery_review_score_max,post_delivery_review_title_count,post_delivery_review_message_count,post_delivery_review_message_length_mean,post_delivery_review_creation_date_min,post_delivery_review_answer_timestamp_max
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,7c396fd4830fd04220f754e42b4e5bff,...,2.0,1.0,4.0,4.0,4.0,0.0,1.0,170.0,2017-10-11,2017-10-12 03:43:48
1,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,True,3a653a41f6f9fc3d2a113cf8398680e8,...,1.0,1.0,5.0,5.0,5.0,0.0,0.0,NaN,2018-08-18,2018-08-22 19:07:58
2,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,True,7c142cf63193a1473d2e66489a9ae977,...,1.0,1.0,5.0,5.0,5.0,0.0,1.0,105.0,2017-12-03,2017-12-05 19:21:58
3,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,True,932afa1e708222e5821dac9cd5db4cae,...,1.0,1.0,5.0,5.0,5.0,0.0,0.0,NaN,2017-05-27,2017-05-28 02:59:57
4,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,True,39382392765b6dc74812866ee5ee92a7,...,1.0,1.0,1.0,1.0,1.0,0.0,0.0,NaN,2017-02-03,2017-02-05 01:58:35


### Review join findings

The review join preserved all 99,441 orders and maintained one row per order.

Review summaries were found for 98,673 orders. A total of 768 orders had no matching review, including 646 delivered orders. This confirms that submitting a review is optional and that a missing review must not be interpreted as a negative review.

Review fields are post-delivery information. They are retained for exploratory analysis of customer satisfaction, but they will be excluded from the prediction-time feature set to prevent target leakage.

## 17. Attach customer geolocation

Customer ZIP-code prefixes are matched with the aggregated geolocation table to add approximate customer latitude and longitude.

The raw geolocation table contains many records per ZIP prefix, so it was first reduced to one representative row per prefix using median latitude and longitude.

This is a many-to-one join:

- many orders may share the same customer ZIP prefix;
- each ZIP prefix has at most one aggregated geolocation record.

The join must preserve every order and maintain one row per `order_id`. Missing coordinate matches are retained and measured rather than silently removed.

In [23]:
customer_geolocation = (
    geolocation_zip_level
    .rename(
        columns={
            "geolocation_zip_code_prefix": (
                "customer_zip_code_prefix"
            ),
            "geolocation_lat_median": (
                "customer_lat_median"
            ),
            "geolocation_lng_median": (
                "customer_lng_median"
            ),
            "geolocation_record_count": (
                "customer_geolocation_record_count"
            ),
        }
    )
    .copy()
)

ml_order_level = ml_order_level.merge(
    customer_geolocation,
    on="customer_zip_code_prefix",
    how="left",
    validate="many_to_one",
    indicator="_customer_geo_merge",
)

print(
    "Rows before customer-geolocation merge:",
    len(orders_customer_level),
)

print(
    "Rows after customer-geolocation merge:",
    len(ml_order_level),
)

print(
    "Orders with customer coordinates:",
    ml_order_level[
        "_customer_geo_merge"
    ].eq("both").sum(),
)

print(
    "Orders without customer coordinates:",
    ml_order_level[
        "_customer_geo_merge"
    ].eq("left_only").sum(),
)

print(
    "Duplicated order IDs:",
    ml_order_level["order_id"].duplicated().sum(),
)

assert len(ml_order_level) == len(orders_customer_level)

assert (
    ml_order_level["order_id"]
    .duplicated()
    .sum()
    == 0
)

missing_customer_zip_summary = (
    ml_order_level.loc[
        ml_order_level["_customer_geo_merge"].eq("left_only"),
        "customer_zip_code_prefix",
    ]
    .value_counts(dropna=False)
    .rename_axis("customer_zip_code_prefix")
    .reset_index(name="order_count")
)

print("\nMissing customer coordinates by ZIP prefix:")
print(
    "Unique unmatched customer ZIP prefixes:",
    len(missing_customer_zip_summary),
)

display(missing_customer_zip_summary.head(10))

ml_order_level = ml_order_level.drop(
    columns="_customer_geo_merge"
)

print(
    "Current ML-table shape:",
    ml_order_level.shape,
)

display(
    ml_order_level[
        [
            "order_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state",
            "customer_lat_median",
            "customer_lng_median",
            "customer_geolocation_record_count",
        ]
    ].head()
)

Rows before customer-geolocation merge: 99441
Rows after customer-geolocation merge: 99441
Orders with customer coordinates: 99163
Orders without customer coordinates: 278
Duplicated order IDs: 0

Missing customer coordinates by ZIP prefix:
Unique unmatched customer ZIP prefixes: 157


,customer_zip_code_prefix,order_count
0,70686,15
1,72005,13
2,71919,10
3,73255,7
4,72300,6
5,71884,5
6,72002,5
7,73401,5
8,73369,5
9,71676,4


Current ML-table shape:

 (99441, 62)


,order_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat_median,customer_lng_median,customer_geolocation_record_count
0,e481f51cbdc54678b7cc49136f2d6af7,03149,sao paulo,SP,-23.576170,-46.587276,24.0
1,47770eb9100c2d0c44946d9cf07ec65d,75265,vianopolis,GO,-16.744472,-48.514624,25.0
2,949d5b44dbf5de918fe9c16f97b45f8a,59296,sao goncalo do amarante,RN,-5.774611,-35.273916,18.0
3,6514b8ad8028c9f2cc2374ded245783f,26525,nilopolis,RJ,-22.805728,-43.423175,163.0
4,76c6e866289321a7c93b82b54852dc33,99655,faxinalzinho,RS,-27.421906,-52.674452,3.0


## 18. Attach primary-seller geolocation

The primary seller ZIP-code prefix is matched with the aggregated geolocation table to add approximate seller latitude and longitude.

This is a many-to-one join because many orders may share the same primary seller ZIP prefix, while the aggregated geolocation table contains at most one row per ZIP prefix.

Missing seller coordinates are separated into:

- orders without a primary seller ZIP prefix;
- orders with a seller ZIP prefix that is not covered by the geolocation table.

Customer-to-seller distance will not be calculated here because distance is an engineered feature and belongs in Notebook 5.

In [24]:
seller_geolocation = (
    geolocation_zip_level
    .rename(
        columns={
            "geolocation_zip_code_prefix": (
                "primary_seller_zip_code_prefix"
            ),
            "geolocation_lat_median": (
                "primary_seller_lat_median"
            ),
            "geolocation_lng_median": (
                "primary_seller_lng_median"
            ),
            "geolocation_record_count": (
                "primary_seller_geolocation_record_count"
            ),
        }
    )
    .copy()
)

ml_order_level = ml_order_level.merge(
    seller_geolocation,
    on="primary_seller_zip_code_prefix",
    how="left",
    validate="many_to_one",
    indicator="_seller_geo_merge",
)

orders_without_seller_zip = (
    ml_order_level[
        "primary_seller_zip_code_prefix"
    ].isna().sum()
)

orders_with_unmatched_seller_zip = (
    ml_order_level[
        "_seller_geo_merge"
    ].eq("left_only")
    &
    ml_order_level[
        "primary_seller_zip_code_prefix"
    ].notna()
).sum()

print(
    "Rows after seller-geolocation merge:",
    len(ml_order_level),
)

print(
    "Orders with seller coordinates:",
    ml_order_level[
        "_seller_geo_merge"
    ].eq("both").sum(),
)

print(
    "Orders without a primary seller ZIP:",
    orders_without_seller_zip,
)

print(
    "Orders with an unmatched seller ZIP:",
    orders_with_unmatched_seller_zip,
)

print(
    "Total orders without seller coordinates:",
    ml_order_level[
        "_seller_geo_merge"
    ].eq("left_only").sum(),
)

print(
    "Duplicated order IDs:",
    ml_order_level["order_id"].duplicated().sum(),
)

assert len(ml_order_level) == len(orders_customer_level)

assert (
    ml_order_level["order_id"]
    .duplicated()
    .sum()
    == 0
)

missing_seller_zip_summary = (
    ml_order_level.loc[
        ml_order_level["_seller_geo_merge"].eq("left_only")
        & ml_order_level[
            "primary_seller_zip_code_prefix"
        ].notna(),
        "primary_seller_zip_code_prefix",
    ]
    .value_counts()
    .rename_axis("primary_seller_zip_code_prefix")
    .reset_index(name="order_count")
)

print("\nUnmatched seller ZIP-prefix summary:")
print(
    "Unique unmatched seller ZIP prefixes:",
    len(missing_seller_zip_summary),
)

display(missing_seller_zip_summary.head(10))

ml_order_level = ml_order_level.drop(
    columns="_seller_geo_merge"
)

print(
    "Current ML-table shape:",
    ml_order_level.shape,
)

display(
    ml_order_level[
        [
            "order_id",
            "primary_seller_zip_code_prefix",
            "primary_seller_city",
            "primary_seller_state",
            "primary_seller_lat_median",
            "primary_seller_lng_median",
            "primary_seller_geolocation_record_count",
        ]
    ].head()
)

Rows after seller-geolocation merge: 99441


Orders with seller coordinates: 98447
Orders without a primary seller ZIP: 775
Orders with an unmatched seller ZIP: 219
Total orders without seller coordinates: 994
Duplicated order IDs: 0

Unmatched seller ZIP-prefix summary:
Unique unmatched seller ZIP prefixes: 7


,primary_seller_zip_code_prefix,order_count
0,02285,127
1,37708,44
2,71551,35
3,82040,8
4,91901,2
5,07412,2
6,72580,1


Current ML-table shape: (99441, 65)


,order_id,primary_seller_zip_code_prefix,primary_seller_city,primary_seller_state,primary_seller_lat_median,primary_seller_lng_median,primary_seller_geolocation_record_count
0,e481f51cbdc54678b7cc49136f2d6af7,09350,maua,SP,-23.681180,-46.444127,207.0
1,47770eb9100c2d0c44946d9cf07ec65d,14840,guariba,SP,-21.364020,-48.228831,170.0
2,949d5b44dbf5de918fe9c16f97b45f8a,31842,belo horizonte,MG,-19.836541,-43.921855,95.0
3,6514b8ad8028c9f2cc2374ded245783f,12940,atibaia,SP,-23.114775,-46.553325,253.0
4,76c6e866289321a7c93b82b54852dc33,13720,sao jose do rio pardo,SP,-21.600754,-46.893793,429.0


### Seller-geolocation findings

The seller-geolocation join preserved all 99,441 orders and maintained one row per order.

Seller coordinates were found for 98,447 orders, giving approximately 99.00% coverage.

A total of 994 orders had no seller coordinates:

- 775 had no primary seller ZIP because no item/seller summary existed;
- 219 had a seller ZIP that was not covered by the geolocation table.

The 219 unmatched records involved only seven seller ZIP prefixes, with most concentrated in prefixes `02285`, `37708`, and `71551`.

Missing seller coordinates are retained and will be handled explicitly during feature engineering.

## 19. Reconcile and deterministically order the joined table

The complete joined table is checked against the original order population.

Structural reconciliation verifies:

- the final row count matches the source order count;
- every `order_id` is present and unique;
- customer identifiers are present;
- no duplicated column names were introduced;
- feature-group coverage agrees with previous join results.

The final table is sorted by `order_id` before saving so repeated executions produce a deterministic row order.

In [25]:
ml_order_level = (
    ml_order_level
    .sort_values(
        by="order_id",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

final_row_count = len(ml_order_level)
unique_order_count = ml_order_level["order_id"].nunique()
duplicated_order_count = (
    ml_order_level["order_id"].duplicated().sum()
)
missing_order_id_count = (
    ml_order_level["order_id"].isna().sum()
)
missing_customer_id_count = (
    ml_order_level["customer_id"].isna().sum()
)
duplicated_column_count = (
    ml_order_level.columns.duplicated().sum()
)

print("Source order rows:", orders_source_count)
print("Final ML-table rows:", final_row_count)
print("Unique order IDs:", unique_order_count)
print("Duplicated order IDs:", duplicated_order_count)
print("Missing order IDs:", missing_order_id_count)
print("Missing customer IDs:", missing_customer_id_count)
print("Duplicated column names:", duplicated_column_count)
print("Final ML-table shape:", ml_order_level.shape)

assert final_row_count == orders_source_count
assert unique_order_count == orders_source_count
assert duplicated_order_count == 0
assert missing_order_id_count == 0
assert missing_customer_id_count == 0
assert duplicated_column_count == 0
assert ml_order_level["order_id"].is_monotonic_increasing

coverage_masks = {
    "Item/product/seller summary": (
        ml_order_level["item_count"].notna()
    ),
    "Payment summary": (
        ml_order_level["payment_record_count"].notna()
    ),
    "Post-delivery review summary": (
        ml_order_level[
            "post_delivery_review_count"
        ].notna()
    ),
    "Customer geolocation": (
        ml_order_level[
            [
                "customer_lat_median",
                "customer_lng_median",
            ]
        ]
        .notna()
        .all(axis=1)
    ),
    "Primary-seller geolocation": (
        ml_order_level[
            [
                "primary_seller_lat_median",
                "primary_seller_lng_median",
            ]
        ]
        .notna()
        .all(axis=1)
    ),
}

coverage_summary = pd.DataFrame(
    [
        {
            "feature_group": feature_group,
            "matched_orders": int(mask.sum()),
            "missing_orders": int((~mask).sum()),
            "coverage_percent": round(
                mask.mean() * 100,
                4,
            ),
        }
        for feature_group, mask in coverage_masks.items()
    ]
)

assert (
    coverage_summary["matched_orders"]
    + coverage_summary["missing_orders"]
).eq(final_row_count).all()

print("\nFeature-group coverage:")

display(coverage_summary)

display(ml_order_level.head())

Source order rows: 99441
Final ML-table rows: 99441
Unique order IDs: 99441
Duplicated order IDs: 0
Missing order IDs: 0
Missing customer IDs: 0
Duplicated column names: 0
Final ML-table shape: (99441, 65)



Feature-group coverage:


,feature_group,matched_orders,missing_orders,coverage_percent
0,Item/product/seller summary,98666,775,99.2206
1,Payment summary,99440,1,99.9990
2,Post-delivery review summary,98673,768,99.2277
3,Customer geolocation,99163,278,99.7204
4,Primary-seller geolocation,98447,994,99.0004


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_record_found,customer_unique_id,...,post_delivery_review_message_count,post_delivery_review_message_length_mean,post_delivery_review_creation_date_min,post_delivery_review_answer_timestamp_max,customer_lat_median,customer_lng_median,customer_geolocation_record_count,primary_seller_lat_median,primary_seller_lng_median,primary_seller_geolocation_record_count
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,True,871766c5855e863f6eccc05f988b23cb,...,1.0,46.0,2017-09-21,2017-09-22 10:57:03,-21.762829,-41.310605,149.0,-22.498419,-44.125272,59.0
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,True,eb28e67c4c0b83846050ddfb8a35d051,...,0.0,NaN,2017-05-13,2017-05-15 11:34:13,-20.205737,-50.926924,367.0,-23.564289,-46.519045,39.0
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,True,3818d81c6709e39d06b2738a8d3a2474,...,1.0,90.0,2018-01-23,2018-01-23 16:06:31,-19.870383,-44.594355,224.0,-22.271648,-46.165556,71.0
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,True,af861d436cfc08b2c2ddefd0ba074622,...,0.0,NaN,2018-08-15,2018-08-15 16:39:01,-23.104346,-46.595589,27.0,-20.554951,-47.387691,438.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,True,64b576fb70d441e8f1b2d7d446e483c5,...,1.0,39.0,2017-03-02,2017-03-03 10:54:59,-23.245512,-46.825182,4.0,-22.930408,-53.136438,119.0


### Final structural reconciliation findings

The complete joined table contains 99,441 rows and 65 columns.

All 99,441 source orders are represented exactly once:

- 99,441 unique order IDs;
- zero duplicated order IDs;
- zero missing order IDs;
- zero missing customer IDs;
- zero duplicated column names.

Coverage counts in the final table agree with the earlier individual join results. The table is sorted by `order_id` to provide deterministic row ordering before artifact creation.

## 20. Audit data types and missing values

The final joined table is inspected for data types and missing values before it is saved.

Missing values are not filled in this notebook. Their meanings differ:

- some result from incomplete or canceled orders;
- some result from absent payment, review, or geolocation records;
- some product attributes were missing in the source data;
- review-message fields may be missing because the customer submitted a score without written text.

Label eligibility will be handled in Notebook 2. Prediction-time feature transformations and imputation will be fitted in Notebook 5 using training data only.

In [26]:
missing_summary = pd.DataFrame(
    {
        "column": ml_order_level.columns,
        "data_type": [
            str(data_type)
            for data_type in ml_order_level.dtypes
        ],
        "non_missing_count": [
            int(ml_order_level[column].notna().sum())
            for column in ml_order_level.columns
        ],
        "missing_count": [
            int(ml_order_level[column].isna().sum())
            for column in ml_order_level.columns
        ],
        "missing_percent": [
            round(
                ml_order_level[column].isna().mean() * 100,
                4,
            )
            for column in ml_order_level.columns
        ],
    }
)

missing_summary = (
    missing_summary
    .sort_values(
        by=["missing_count", "column"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

columns_with_missing_values = missing_summary.loc[
    missing_summary["missing_count"] > 0
].copy()

complete_columns = missing_summary.loc[
    missing_summary["missing_count"] == 0
].copy()

print("Total columns:", ml_order_level.shape[1])

print(
    "Columns with missing values:",
    len(columns_with_missing_values),
)

print(
    "Columns without missing values:",
    len(complete_columns),
)

assert (
    len(columns_with_missing_values)
    + len(complete_columns)
    == ml_order_level.shape[1]
)

assert (
    missing_summary["non_missing_count"]
    + missing_summary["missing_count"]
).eq(len(ml_order_level)).all()

print("\nColumns containing missing values:")

display(columns_with_missing_values)

print("\nComplete columns:")

display(
    complete_columns[
        [
            "column",
            "data_type",
            "non_missing_count",
        ]
    ]
)

Total columns: 65
Columns with missing values: 55
Columns without missing values: 10

Columns containing missing values:


,column,data_type,non_missing_count,missing_count,missing_percent
0,post_delivery_review_message_length_mean,float64,40827,58614,58.9435
1,order_delivered_customer_date,datetime64[us],96476,2965,2.9817
2,product_description_length_mean,float64,97277,2164,2.1762
3,product_name_length_mean,float64,97277,2164,2.1762
4,product_photos_qty_mean,float64,97277,2164,2.1762
5,order_delivered_carrier_date,datetime64[us],97658,1783,1.7930
6,primary_seller_geolocation_record_count,float64,98447,994,0.9996
7,primary_seller_lat_median,float64,98447,994,0.9996
8,primary_seller_lng_median,float64,98447,994,0.9996
9,product_height_cm_mean,float64,98650,791,0.7954



Complete columns:


,column,data_type,non_missing_count
55,customer_city,str,99441
56,customer_id,str,99441
57,customer_record_found,bool,99441
58,customer_state,str,99441
59,customer_unique_id,str,99441
60,customer_zip_code_prefix,str,99441
61,order_estimated_delivery_date,datetime64[us],99441
62,order_id,str,99441
63,order_purchase_timestamp,datetime64[us],99441
64,order_status,str,99441


### Missing-value findings

The joined table has 65 columns: 55 contain at least one missing value and 10 are complete.

The largest missingness rate belongs to `post_delivery_review_message_length_mean` (58.9435%) because many customers submit a score without a written message. This is meaningful optional behaviour, not evidence of a failed join.

The 2,965 missing customer-delivery timestamps primarily represent orders without a completed delivery outcome. Notebook 2 will determine which orders have sufficient information for a trustworthy late-delivery label.

Other missing-value groups reconcile with earlier checks:

- 775 orders lack item/product/seller summaries;
- 768 orders lack post-delivery review summaries;
- 278 orders lack customer coordinates;
- 994 orders lack primary-seller coordinates;
- one delivered order lacks a payment summary.

No missing values are imputed here. Keeping the source missingness intact preserves provenance and prevents transformations from being fitted before the train/validation/test split.


## 21. Save and verify the joined-table artifact

The deterministically ordered table is saved as a Parquet artifact for Notebook 2.

Parquet is used because it preserves column data types, supports efficient storage, and avoids repeated database joins in downstream notebooks. The saved file is immediately reloaded and compared with the in-memory table to verify that the round trip changed neither values nor data types.

A SHA-256 digest is printed as a provenance identifier for the exact artifact bytes produced by this environment.


In [27]:
import hashlib

ml_order_level.to_parquet(
    OUTPUT_PATH,
    index=False,
    engine="pyarrow",
)

assert OUTPUT_PATH.exists(), "The joined-table artifact was not created."

reloaded_order_level = pd.read_parquet(
    OUTPUT_PATH,
    engine="pyarrow",
)

pd.testing.assert_frame_equal(
    ml_order_level,
    reloaded_order_level,
    check_dtype=True,
    check_like=False,
)

artifact_size_bytes = OUTPUT_PATH.stat().st_size
artifact_sha256 = hashlib.sha256(
    OUTPUT_PATH.read_bytes()
).hexdigest()

print("Artifact saved successfully.")
print(f"Path: {OUTPUT_PATH}")
print(f"Rows: {len(reloaded_order_level):,}")
print(f"Columns: {reloaded_order_level.shape[1]}")
print(f"Size: {artifact_size_bytes / (1024 ** 2):.2f} MiB")
print(f"SHA-256: {artifact_sha256}")
print("Reloaded artifact exactly matches the in-memory table.")


Artifact saved successfully.
Path: D:\Documents\mlops-olist\data\processed\orders_joined.parquet
Rows: 99,441
Columns: 65
Size: 20.96 MiB
SHA-256: ea4fa493fbe3c712ca356ccc2007f3eba8b044ab1e7b0ada65a974a3546b4a27
Reloaded artifact exactly matches the in-memory table.


## 22. Notebook 1 completion summary

Notebook 1 produced `data/processed/orders_joined.parquet`, an analysis-ready table with:

- 99,441 rows;
- 65 columns;
- exactly one row per `order_id`;
- deterministic ordering by `order_id`;
- aggregated item, product, seller, payment, review, and ZIP-prefix geography data;
- documented coverage and missingness;
- verified Parquet round-trip equality.

The table intentionally retains all order statuses and source missing values. It does not create the target and does not impute features.

### Handoff to Notebook 2

Notebook 2 will read this artifact, define which orders have a valid delivery outcome, create the late-delivery label by comparing actual and estimated delivery timestamps, validate examples manually, and measure class balance.

Post-delivery review fields and other future-only information must never be used as prediction-time model inputs.
